# 07 · SAINT+: Kuizi's brain
**Kuizi brain walkthrough, notebook 7 of 10**

**SAINT+** (Separated Self-AttentIve Neural Knowledge Tracing, 2021) is the model Kuizi's brain is
built on. It is a **transformer** with two halves, kept separate on purpose:

| | Encoder: "what was asked" | Decoder: "how the student answered" |
| --- | --- | --- |
| Reads | the items and their concepts | the previous answers, how long each took (**elapsed time**), how long since the last answer (**lag time**) |
| Inside | causal self-attention over past items | causal self-attention over past answers, then **cross-attention** to the encoder |
| Gives | a representation of each question in context | the chance that the student answers the current question correctly |

Each item, concept and answer is first turned into a vector of numbers, an **embedding**, that
the model learns. A **position** embedding tells the model the order of events.

| In this notebook | Production ticket |
| --- | --- |
| Production SAINT+ with elapsed and lag time | T042 |
| Model benchmark that picks the production model | T040 |
| pyKT, the research library with reference implementations | T041 |

<div dir="rtl" lang="ar">

## ٠٧ · SAINT+: دماغ كويزي

<b>SAINT+</b> هو النموذج الذي يُبنى عليه دماغ كويزي. إنه <b>محوِّل</b> (transformer) بنصفين منفصلين عن قصد:
<b>المُرمِّز</b> يقرأ "ما الذي سُئل": التمارين ومفاهيمها. و<b>مُفكِّك الترميز</b> يقرأ "كيف أجاب التلميذ": الإجابات السابقة،
و<b>الزمن المستغرق</b> في كل إجابة، و<b>الفاصل الزمني</b> منذ آخر إجابة، ثم ينظر إلى المُرمِّز عبر <b>الانتباه المتقاطع</b>
ليعطي احتمال أن يجيب التلميذ عن السؤال الحالي إجابة صحيحة.

كل تمرين ومفهوم وإجابة يتحوّل أولًا إلى متّجه أرقام يتعلّمه النموذج يسمّى <b>التضمين</b> (embedding)، وتضمين <b>الموضع</b>
يخبر النموذج بترتيب الأحداث.

</div>

In [ ]:
# Concept: model settings · Production: T042
EPOCHS, BATCH, LR = 6, 16, 1e-3   # SAINT+ training settings (see notebook 7)

In [ ]:
# Concept: following one batch through SAINT+ · Production: T042
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from kuizi_brain import data as D, models as M, metrics as X
torch.manual_seed(0)
sim = D.simulate_students(n_students=1500, answers_per_student=120, seed=7)
train, test = D.split_by_student(sim)
w_train, w_test = M.make_windows(train), M.make_windows(test)
model = M.SaintPlus(sim.attrs["n_items"], sim.attrs["n_kc"], d=64, heads=4, layers=2)
batch = {k: v[:2] for k, v in w_train.items()}
print("input: 2 windows of", tuple(batch["item"].shape), "steps")
print("item embedding  :", tuple(model.item(batch["item"]).shape), "  (each item becomes 64 numbers)")
print("concept embedding:", tuple(model.kc(batch["kc"]).shape))
print("output logits    :", tuple(model(batch).shape), "  (one chance of success per step)")
print(f"parameters to learn: {M.count_parameters(model):,}")

## Training

The training loop shows the model windows of 100 answers, compares its predictions with what
students really answered (the **loss**), and adjusts the 300,000-odd numbers a little each time.
One pass over all training windows is an **epoch**.

<div dir="rtl" lang="ar">

## التدريب

حلقة التدريب تعرض على النموذج نوافذ من 100 إجابة، وتقارن تنبّؤاته بما أجاب به التلاميذ فعلًا (<b>الخسارة</b>)،
وتعدّل أرقامه قليلًا في كل مرّة. المرور الكامل على كل نوافذ التدريب يسمّى <b>حقبة</b> (epoch).

</div>

In [ ]:
# Concept: training SAINT+ with time features · Production: T042
saint = M.SaintPlus(sim.attrs["n_items"], sim.attrs["n_kc"])
history = M.fit(saint, w_train, epochs=EPOCHS, batch_size=BATCH, lr=LR)
y, p_saint = M.predict(saint, w_test)

In [ ]:
# Concept: comparing with DKT and with the hidden truth · Production: T040
dkt = M.DKT(sim.attrs["n_kc"])
M.fit(dkt, w_train, epochs=EPOCHS, batch_size=BATCH, lr=LR, verbose=False)
_, p_dkt = M.predict(dkt, w_test)
rows = [X.report("SAINT+", y, p_saint), X.report("DKT", y, p_dkt), X.report("hidden truth (ceiling)", y, test.p_true.values)]
table = pd.DataFrame(rows)
table["corr. with truth"] = [np.corrcoef(p_saint, test.p_true)[0, 1].round(3), np.corrcoef(p_dkt, test.p_true)[0, 1].round(3), 1.0]
table

In [ ]:
# Concept: what the numbers say · Production: T040
gap = table.AUC[0] - table.AUC[1]
leader = "SAINT+" if gap > 0 else "DKT"
print(f"On this simulated data {leader} leads by {abs(gap):.3f} AUC.")
print(f"Both stay below the ceiling of {table.AUC[2]:.3f}: no model can predict luck.")

## Do the time features help?

We train the same model without elapsed and lag time and compare. In the simulator, a slow
answer tends to mean an uncertain student, and long breaks cause forgetting.

<div dir="rtl" lang="ar">

## هل تساعد ميزات الزمن؟

ندرّب نفس النموذج دون الزمن المستغرق والفاصل الزمني ونقارن. في المحاكي، الإجابة البطيئة تعني غالبًا تلميذًا غير متأكّد،
والانقطاع الطويل يسبّب النسيان.

</div>

In [ ]:
# Concept: SAINT+ without time features · Production: T042
saint_no_time = M.SaintPlus(sim.attrs["n_items"], sim.attrs["n_kc"], use_time=False)
M.fit(saint_no_time, w_train, epochs=EPOCHS, batch_size=BATCH, lr=LR, verbose=False)
_, p_nt = M.predict(saint_no_time, w_test)
print(f"with time features   : AUC {X.auc(y, p_saint):.3f}")
print(f"without time features: AUC {X.auc(y, p_nt):.3f}")

## On real data, small is hard

On ASSISTments (about 260,000 answers, 15,000 items) SAINT+ has many item embeddings to learn
from few answers each. Published benchmarks find the same: SAINT-type models shine on very
large datasets (millions of answers, like EdNet) and do not beat DKT on small ones. This is why
the choice of Kuizi's production model is made by a benchmark on real data (T040, then T239 on
pilot data), not by reputation.

<div dir="rtl" lang="ar">

## على البيانات الحقيقية، الصغير صعب

في ASSISTments (نحو 260,000 إجابة و15,000 تمرين) على SAINT+ أن يتعلّم تضمينات كثيرة للتمارين من إجابات قليلة لكل منها.
والدراسات المنشورة تجد نفس الشيء: نماذج SAINT تتألّق على البيانات الضخمة جدًا (ملايين الإجابات مثل EdNet) ولا تتفوّق
على DKT في البيانات الصغيرة. لهذا يُختار نموذج كويزي النهائي بمقارنة على بيانات حقيقية (T040 ثم T239 على بيانات التجربة)، لا بالسمعة.

</div>

In [ ]:
# Concept: SAINT+ vs DKT on ASSISTments 2009 · Production: T040, T041
real = D.load_assist2009()
r_train, r_test = D.split_by_student(real)
rw_train, rw_test = M.make_windows(r_train), M.make_windows(r_test)
s_real = M.SaintPlus(real.item.max() + 1, real.kc.max() + 1)
M.fit(s_real, rw_train, epochs=4, batch_size=32, lr=LR, verbose=False)
d_real = M.DKT(real.kc.max() + 1)
M.fit(d_real, rw_train, epochs=4, batch_size=32, lr=2e-3, verbose=False)
pd.DataFrame([X.report("SAINT+", *M.predict(s_real, rw_test)), X.report("DKT", *M.predict(d_real, rw_test))])

### Remember
- SAINT+ = encoder over questions + decoder over answers and time, linked by cross-attention.
- Embeddings turn items, concepts and answers into vectors the model learns; positions give the order.
- It predicts, for each new question, the chance of a right answer from the whole history.
- Bigger is not automatically better: the benchmark on real data decides (T040, T239).

<div dir="rtl" lang="ar">

### تذكّر
- SAINT+ = مُرمِّز للأسئلة + مُفكِّك ترميز للإجابات والزمن، يربط بينهما الانتباه المتقاطع.
- التضمينات تحوّل التمارين والمفاهيم والإجابات إلى متّجهات يتعلّمها النموذج، والمواضع تعطي الترتيب.
- يتنبّأ، لكل سؤال جديد، باحتمال الإجابة الصحيحة انطلاقًا من التاريخ كلّه.
- الأكبر ليس الأفضل تلقائيًا: المقارنة على بيانات حقيقية هي التي تحسم (T040، T239).

</div>